In [115]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer, SimpleImputer

TARGET_COL = "y"
TEST_SIZE = 0.2
RANDOM_STATE = 42

df = pd.read_csv(r"data/data.csv", sep=";")
print(f"Dataset shape: {df.shape}")
df.head(3)

X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].copy()


Dataset shape: (45211, 17)


In [116]:
X.drop(columns=["month", "day"], inplace=True)

In [117]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numerical columns:   {len(numerical_cols)}")

Train shape: (36168, 14)
Test shape:  (9043, 14)
Categorical columns: 8
Numerical columns:   6


In [118]:
X_train["previous"] = np.log1p(X_train["previous"])
X_test["previous"] = np.log1p(X_test["previous"])

In [119]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# copy to avoid overwriting
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# numerical columns only
num_cols = X_train_scaled.select_dtypes(include=["number"]).columns

# Standard scaling
std_scaler = StandardScaler()

X_train_scaled[num_cols] = std_scaler.fit_transform(X_train_scaled[num_cols])
X_test_scaled[num_cols] = std_scaler.transform(X_test_scaled[num_cols])

# MinMax scaling (after standard OR instead of — depends what you want)
minmax_scaler = MinMaxScaler()

X_train[num_cols] = minmax_scaler.fit_transform(X_train_scaled[num_cols])
X_test[num_cols] = minmax_scaler.transform(X_test_scaled[num_cols])

X_train.head(3)

,age,job,marital,education,default,balance,housing,loan,contact,duration,campaign,pdays,previous,poutcome
24001,0.233766,technician,divorced,secondary,no,0.080620,no,no,telephone,0.028467,0.016129,0.000000,0.000000,unknown
43409,0.077922,student,single,secondary,no,0.110263,no,no,cellular,0.184425,0.048387,0.213303,0.369981,failure
20669,0.337662,technician,single,secondary,no,0.075019,yes,no,cellular,0.352786,0.048387,0.000000,0.000000,unknown


In [120]:
# 1) Turn 'unknown' into NaN for categorical columns
for col in categorical_cols:
    X_train[col] = X_train[col].replace("unknown", np.nan)
    X_test[col] = X_test[col].replace("unknown", np.nan)


In [121]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import zscore, zmap
import numpy as np
import pandas as pd

class ZScoreOutlierRemover(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None, percentile=99.9):
        self.columns = columns
        self.percentile = percentile

    def fit(self, X, y=None):
        X = X.copy()

        if self.columns is None:
            self.columns_ = X.select_dtypes(include=[np.number]).columns.tolist()
        else:
            self.columns_ = list(self.columns)

        self.zscore_info_ = {}

        for col in self.columns_:
            train_series = X[col]
            mean_col = train_series.mean()
            std_col = train_series.std(ddof=0)

            if pd.isna(std_col) or std_col == 0:
                self.zscore_info_[col] = {
                    "mean": mean_col,
                    "std": std_col,
                    "cutoff": np.inf,
                    "train_removed": 0,
                    "test_removed": 0,
                    "train_pct": 0.0,
                    "test_pct": 0.0,
                }
                continue

            abs_z_train = pd.Series(
                np.abs(zscore(train_series, ddof=0, nan_policy="omit")),
                index=train_series.index
            )

            cutoff = np.nanpercentile(abs_z_train, self.percentile)

            self.zscore_info_[col] = {
                "mean": mean_col,
                "std": std_col,
                "cutoff": cutoff,
                "train_removed": None,
                "test_removed": None,
                "train_pct": None,
                "test_pct": None,
            }

        return self

    def transform(self, X):
        X = X.copy()

        for col in self.columns_:
            info = self.zscore_info_[col]
            mean_col = info["mean"]
            std_col = info["std"]
            cutoff = info["cutoff"]

            if pd.isna(std_col) or std_col == 0 or np.isinf(cutoff):
                continue

            # Use training mean/std implicitly through zmap against training distribution stats
            # reconstructed from mean/std is not possible directly, so we compute manually here
            abs_z = np.abs((X[col] - mean_col) / std_col)

            X.loc[abs_z > cutoff, col] = np.nan

        return X

    def fit_transform_train_test(self, X_train, X_test):
        X_train = X_train.copy()
        X_test = X_test.copy()

        self.fit(X_train)

        for col in self.columns_:
            train_series = X_train[col]
            test_series = X_test[col]

            mean_col = self.zscore_info_[col]["mean"]
            std_col = self.zscore_info_[col]["std"]
            cutoff = self.zscore_info_[col]["cutoff"]

            if pd.isna(std_col) or std_col == 0 or np.isinf(cutoff):
                self.zscore_info_[col]["train_removed"] = 0
                self.zscore_info_[col]["test_removed"] = 0
                self.zscore_info_[col]["train_pct"] = 0.0
                self.zscore_info_[col]["test_pct"] = 0.0
                continue

            abs_z_train = pd.Series(
                np.abs(zscore(train_series, ddof=0, nan_policy="omit")),
                index=train_series.index
            )
            train_outliers = (abs_z_train > cutoff).sum()
            X_train.loc[abs_z_train > cutoff, col] = np.nan

            abs_z_test = pd.Series(
                np.abs((test_series - mean_col) / std_col),
                index=test_series.index
            )
            test_outliers = (abs_z_test > cutoff).sum()
            X_test.loc[abs_z_test > cutoff, col] = np.nan

            self.zscore_info_[col]["train_removed"] = int(train_outliers)
            self.zscore_info_[col]["test_removed"] = int(test_outliers)
            self.zscore_info_[col]["train_pct"] = train_outliers / len(X_train) * 100
            self.zscore_info_[col]["test_pct"] = test_outliers / len(X_test) * 100

        return X_train, X_test
    
# 2) Mark top 0.1% absolute z-score values as outliers (set to NaN)

zscore_transformer = ZScoreOutlierRemover(columns=numerical_cols, percentile=99.9)

X_train, X_test = zscore_transformer.fit_transform_train_test(X_train, X_test)

zscore_info = zscore_transformer.zscore_info_

# Convert to readable table
zscore_summary = pd.DataFrame(zscore_info).T

print("\nOutlier removal summary:")
display(zscore_summary.sort_values("train_removed", ascending=False))


Outlier removal summary:


,mean,std,cutoff,train_removed,test_removed,train_pct,test_pct
balance,0.085200,0.027858,11.141140,37.0,3.0,0.102300,0.033175
duration,0.052563,0.052692,7.285615,37.0,7.0,0.102300,0.077408
age,0.297312,0.138012,3.962293,36.0,5.0,0.099536,0.055291
pdays,0.047199,0.114864,6.098595,36.0,8.0,0.099536,0.088466
previous,0.040125,0.094821,5.460335,36.0,9.0,0.099536,0.099524
campaign,0.028451,0.050066,9.418478,29.0,9.0,0.080181,0.099524


In [122]:
missing_before_imputation_train = X_train.isna().sum().sum()
missing_before_imputation_test = X_test.isna().sum().sum()

print(f"\nMissing values in train before imputation: {missing_before_imputation_train}")
print(f"Missing values in test before imputation:  {missing_before_imputation_test}")


Missing values in train before imputation: 41902
Missing values in test before imputation:  10474


In [123]:
# Split numeric and categorical parts
X_train_num = X_train[numerical_cols].copy()
X_test_num = X_test[numerical_cols].copy()

X_train_cat = X_train[categorical_cols].copy()
X_test_cat = X_test[categorical_cols].copy()

# Impute numeric columns with KNN
num_imputer = KNNImputer(n_neighbors=5, weights="distance")

X_train_num_imputed = pd.DataFrame(
    num_imputer.fit_transform(X_train_num),
    columns=numerical_cols,
    index=X_train_num.index,
)

X_test_num_imputed = pd.DataFrame(
    num_imputer.transform(X_test_num),
    columns=numerical_cols,
    index=X_test_num.index,
)

# Impute categorical columns with most frequent
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_cat_imputed = pd.DataFrame(
    cat_imputer.fit_transform(X_train_cat),
    columns=categorical_cols,
    index=X_train_cat.index,
)

X_test_cat_imputed = pd.DataFrame(
    cat_imputer.transform(X_test_cat),
    columns=categorical_cols,
    index=X_test_cat.index,
)


In [124]:
from sklearn.preprocessing import OneHotEncoder

# One-hot encode categorical columns
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")

X_train_cat_encoded = encoder.fit_transform(X_train_cat_imputed)
X_test_cat_encoded = encoder.transform(X_test_cat_imputed)


enconded_cols = encoder.get_feature_names_out(categorical_cols)

X_train_cat_encoded = pd.DataFrame(
    X_train_cat_encoded,
    columns=enconded_cols,
    index=X_train_cat_imputed.index,
)

X_test_cat_encoded = pd.DataFrame(
    X_test_cat_encoded,
    columns=enconded_cols,
    index=X_test_cat_imputed.index,
)

#X_train_cat_encoded.head(3)

In [125]:
# Recombine processed features
X_train_processed = pd.concat([X_train_num_imputed, X_train_cat_encoded], axis=1)
X_test_processed = pd.concat([X_test_num_imputed, X_test_cat_encoded], axis=1)

print(f"Missing values in train after imputation: {X_train_processed.isna().sum().sum()}")
print(f"Missing values in test after imputation:  {X_test_processed.isna().sum().sum()}")

Missing values in train after imputation: 0
Missing values in test after imputation:  0


In [126]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

le = LabelEncoder()

y_train = pd.Series(
    le.fit_transform(y_train),
    index=y_train.index,
    name=y_train.name
)

y_test = pd.Series(
    le.transform(y_test),
    index=y_test.index,
    name=y_test.name
)

print(dict(zip(le.classes_, le.transform(le.classes_))))

{'no': np.int64(0), 'yes': np.int64(1)}


In [127]:
from imblearn.under_sampling import InstanceHardnessThreshold
from collections import Counter

print("Before undersampling:")
print("Train:", Counter(y_train))
print("Test :", Counter(y_test))

undersampler = InstanceHardnessThreshold(
    random_state=42
)

X_train_processed, y_train = undersampler.fit_resample(X_train_processed, y_train)

print("\nAfter undersampling:")
print("Train:", Counter(y_train))
print("Test (unchanged):", Counter(y_test))
print("X_train shape:", X_train_processed.shape)
print("X_test shape :", X_train_processed.shape)

Before undersampling:
Train: Counter({0: 31937, 1: 4231})
Test : Counter({0: 7985, 1: 1058})

After undersampling:
Train: Counter({0: 8608, 1: 4231})
Test (unchanged): Counter({0: 7985, 1: 1058})
X_train shape: (12839, 26)
X_test shape : (12839, 26)


In [128]:
X_train_scaled

,age,job,marital,education,default,balance,housing,loan,contact,duration,campaign,pdays,previous,poutcome
24001,-0.460434,technician,divorced,secondary,no,-0.164410,no,no,telephone,-0.457311,-0.246104,-0.410910,-0.423167,unknown
43409,-1.589641,student,single,secondary,no,0.899627,no,no,cellular,2.502493,0.398202,1.446096,3.478737,failure
20669,0.292371,technician,single,secondary,no,-0.365486,yes,no,cellular,5.697691,0.398202,-0.410910,-0.423167,unknown
18810,0.668773,unemployed,married,secondary,no,-0.445003,no,no,telephone,-0.862499,2.653271,-0.410910,-0.423167,unknown
23130,-0.272233,technician,married,secondary,no,-0.361249,no,no,cellular,-0.777602,2.331118,-0.410910,-0.423167,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17958,0.856974,blue-collar,married,secondary,no,-0.146160,yes,no,cellular,-0.773743,-0.246104,-0.410910,-0.423167,unknown
15941,-0.460434,admin.,single,secondary,no,-0.437834,yes,no,cellular,-0.700423,0.720354,-0.410910,-0.423167,unknown
16952,0.386471,blue-collar,married,secondary,no,-0.419258,yes,no,cellular,-0.619386,-0.568256,-0.410910,-0.423167,unknown
34781,-1.307339,management,single,tertiary,no,0.388954,yes,no,cellular,-0.121583,-0.568256,-0.410910,-0.423167,unknown


In [129]:
print("NaNs in train:", X_train_scaled.isna().sum().sum())

NaNs in train: 0


In [130]:
# save data for next steps
X_train_processed.to_csv("data/X_train_processed.csv", index=False)
X_test_processed.to_csv("data/X_test_processed.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False)
y_test.to_csv("data/y_test.csv", index=False)